# Lab L1.1 · Customer master data-quality remediation
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 1 · ⏱ 90 min · **Level 2 · Guided starter (fill in the blanks) · ★ recommended level**

Take a 257k-row dirty customer extract to a validated, PII-free, model-ready table with an auditable data-quality report.

### How this notebook works
- Every `✏️ BLANK n` marks code **you** write. Inline blanks look like `____`; block blanks say `# ...your code here...`.
- The markdown cell above each code cell explains **what** to write, **where**, and **why**.
- Stuck > 5 min? Open the 💡 hint in the HTML lab guide, then ask your trainer. The Level 3 solution is released after the lab.

### Scenario
**Horizon Gulf Bank (HGB)** is a fictional UAE retail bank with 250,000 customers. Its Board approved **Project MIZAN** — an AI decisioning platform covering credit, fraud, collateral valuation, cash logistics, customer retention and document automation. You are a junior ML practitioner in the MIZAN squad.

The customer master is assembled nightly from three systems (core banking, CRM, cards). Analysts complain about duplicate customers, *DXB vs Dubai*, salaries stored as text and impossible ages. Every MIZAN model (churn, segmentation, fraud enrichment) will read **your** cleaned table, so it must be correct, documented and repeatable.

**Progressive architecture:** your `customers_clean.parquet` is the input to Lab L3.2 (segmentation) and enriches later labs.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Load and profile the raw extract

**Why this matters.** You cannot fix what you have not measured. A reusable **profile function** turns ‘the data looks messy’ into numbers you can track before and after cleaning — and later in production monitoring (Day 5).

#### ✏️ What to write in the next cell
- **Blank 1** *(inline)* — share of NaN per column, as a percentage
- **Blank 2** *(inline)* — count rows identical in every column
- **Blank 3** *(inline)* — count repeated IDs (entity-level duplicates)

In [ ]:
raw = pd.read_csv(DATA / "customers_raw.csv", low_memory=False)

def profile(df: pd.DataFrame) -> pd.DataFrame:
    """One row per column: dtype, missing %, cardinality and a few example values."""
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_pct": ____,  # ✏️ B1
        "n_unique": df.nunique(dropna=True),
        "examples": [df[c].dropna().astype(str).unique()[:3].tolist() for c in df.columns],
    })

prof_raw = profile(raw)
print(raw.shape)
print("exact duplicate rows   :", ____)  # ✏️ B2
print("duplicate customer_ids :", ____)  # ✏️ B3
prof_raw.sort_values("missing_pct", ascending=False).head(12)

**✔ Checkpoint.** `(257500, 28)`, ~5,000 exact duplicates and ~7,500 duplicated IDs. `account_closed_date` is the most-missing column.

## Step 2 · Remove PII and post-outcome (leakage) columns

**Why this matters.** **PII** (email, mobile) has no predictive legitimacy and creates privacy risk (UAE PDPL, CBUAE consumer protection). **Post-outcome** columns are recorded *after* churn happens — keeping them would leak the answer. Prove it with numbers, then drop them.

#### ✏️ What to write in the next cell
- **Blank 4** *(inline)* — drop every key of the dict; returns a new DataFrame so raw stays intact

In [ ]:
print(raw.groupby("retention_offer_made").churned_6m.mean().round(3))
print(raw.groupby(raw.account_closed_date.notna()).churned_6m.mean().round(3))

dropped = {
    "email": "PII — direct identifier, no legitimate predictive value",
    "mobile": "PII — direct identifier",
    "account_closed_date": "LEAKAGE — only populated after the customer has churned",
    "retention_offer_made": "LEAKAGE — offer is triggered by the churn process itself",
}
df = ____  # ✏️ B4
print(df.shape)

**✔ Checkpoint.** `account_closed_date` filled ⇒ churn = 1.0 (perfect leak); offer made ⇒ churn ≈ 0.7 vs ≈ 0.05. Shape becomes `(257500, 24)`.

## Step 3 · Standardise categorical values

**Why this matters.** `DXB`, `dubai ` and `DUBAI` are one emirate. If we de-duplicate before standardising, near-duplicates survive. Mapping to a **controlled vocabulary** and asserting the result is how production pipelines stay consistent.

#### ✏️ What to write in the next cell
- **Blank 5** *(code block, ≈2 lines)* — normalise the text (strip, upper-case, remove '-' and spaces) then map with EMIRATE_MAP

In [ ]:
EMIRATE_MAP = {"DUBAI": "Dubai", "DXB": "Dubai", "ABUDHABI": "Abu Dhabi", "AUH": "Abu Dhabi",
               "SHARJAH": "Sharjah", "SHJ": "Sharjah", "AJMAN": "Ajman", "AJM": "Ajman",
               "RASALKHAIMAH": "Ras Al Khaimah", "RAK": "Ras Al Khaimah", "FUJAIRAH": "Fujairah", "FUJ": "Fujairah",
               "UMMALQUWAIN": "Umm Al Quwain", "UAQ": "Umm Al Quwain"}

def std_emirate(s: pd.Series) -> pd.Series:
    # ✏️ BLANK 5 — write ≈2 line(s): normalise the text (strip, upper-case, remove '-' and spaces) then map with EMIRATE_MAP
    # ...your code here...

df["emirate"] = std_emirate(df["emirate"])
df["gender"] = df["gender"].astype(str).str.strip().str.upper().str[0].map({"M": "M", "F": "F"})
assert set(df.emirate.dropna()) == set(EMIRATE_MAP.values()) and df.emirate.notna().all()
assert set(df.gender.unique()) == {"M", "F"}
df.emirate.value_counts()

**✔ Checkpoint.** Exactly 7 emirates; Dubai ≈ 40%, Abu Dhabi ≈ 28%. Both asserts pass silently.

## Step 4 · Parse messy numerics and mixed-format dates

**Why this matters.** Source systems disagree on formats: `AED 12,500` vs `12500.0`; `2019-03-04` vs `04/03/2019` vs `Mar 04 2019`. Parsing must be **explicit** (UAE dates are day-first) and every failure must be counted, never silently dropped.

#### ✏️ What to write in the next cell
- **Blank 6** *(inline)* — regex for every char that is NOT a digit or a dot
- **Blank 7** *(inline)* — let pandas infer each row's format
- **Blank 8** *(inline)* — UAE convention: 04/03/2019 is 4 March
- **Blank 9** *(code block, ≈2 lines)* — fill missing tenure_months from customer_since (months ≈ days / 30.4), as integers

In [ ]:
sal_txt = df["monthly_salary_aed"].astype(str)
df["monthly_salary_aed"] = pd.to_numeric(sal_txt.str.replace(____, "", regex=True), errors="coerce")  # ✏️ B6
was_missing = raw["monthly_salary_aed"].isna().sum()
print("salary parse failures (excluding genuinely missing):", df.monthly_salary_aed.isna().sum() - was_missing)

df["customer_since"] = pd.to_datetime(df["customer_since"], format=____, dayfirst=____)  # ✏️ B7; B8
assert df.customer_since.notna().all(), "unparsed dates!"

ref = pd.Timestamp("2025-10-31")
# ✏️ BLANK 9 — write ≈2 line(s): fill missing tenure_months from customer_since (months ≈ days / 30.4), as integers
# ...your code here...
df[["monthly_salary_aed", "customer_since", "tenure_months"]].describe()

**✔ Checkpoint.** 0 salary parse failures; all dates parsed; `tenure_months` has no missing values.

## Step 5 · Validate ranges and remove entity-level duplicates

**Why this matters.** Validity rules encode business knowledge: customers are 18–100, balances are non-negative for this product set. Invalid ≠ delete — set to missing so the row’s other information survives. Then **de-duplicate on the entity key**, keeping the most complete record.

#### ✏️ What to write in the next cell
- **Blank 10** *(code block, ≈4 lines)* — keep the most complete row per customer_id: count NaNs per row, sort ascending, drop duplicate IDs, remove helper column

In [ ]:
bad_age = ~df.age.between(18, 100)
bad_bal = df.avg_balance_aed < 0
print(f"invalid ages: {bad_age.sum()}, negative balances: {bad_bal.sum()}")
df.loc[bad_age, "age"] = np.nan
df.loc[bad_bal, "avg_balance_aed"] = np.nan

n0 = len(df)
df = df.drop_duplicates()
print("exact duplicates removed:", n0 - len(df))
# ✏️ BLANK 10 — write ≈4 line(s): keep the most complete row per customer_id: count NaNs per row, sort ascending, drop duplicate IDs, remove helper column
# ...your code here...
assert df.customer_id.is_unique and len(df) == 250_000
print("unique customers:", len(df))

**✔ Checkpoint.** ~1,000 invalid ages, ~500 negative balances, ~5,000 exact duplicates removed, then 250,000 unique customers.

## Step 6 · Impute missing values by mechanism

**Why this matters.** Not all missing values are equal. `aecb_score` is missing mostly for **new-to-country** customers (MNAR) — the *absence* is informative, so we add a flag. Salary missingness looks random within groups (MAR) — a **group median** is more accurate than a global one. Final model-time imputation happens inside pipelines; here we fix what is safe and add indicators.

#### ✏️ What to write in the next cell
- **Blank 11** *(inline)* — 1 when the bureau score is absent — keeps the MNAR signal
- **Blank 12** *(inline)* — the two columns that best explain salary level

In [ ]:
print(df.groupby(df.aecb_score.isna())[["tenure_months", "churned_6m"]].agg(["median", "mean"]).round(3))

df["aecb_missing"] = ____  # ✏️ B11
df["salary_missing"] = df.monthly_salary_aed.isna().astype(int)
df["employer_category"] = df["employer_category"].fillna("Unknown")

grp_median = df.groupby(____)["monthly_salary_aed"].transform("median")  # ✏️ B12
df["monthly_salary_aed"] = df["monthly_salary_aed"].fillna(grp_median).fillna(df.monthly_salary_aed.median())
df["digital_logins_30d"] = df["digital_logins_30d"].fillna(df.digital_logins_30d.median())
df["age"] = df["age"].fillna(df.groupby("employment_type")["age"].transform("median"))
df["avg_balance_aed"] = df["avg_balance_aed"].fillna(df.groupby("employer_category")["avg_balance_aed"].transform("median"))

remaining = df.isna().sum(); print(remaining[remaining > 0])

**✔ Checkpoint.** Customers without a bureau score have much shorter tenure (MNAR). Afterwards only `aecb_score` still has NaNs — on purpose, for the model pipeline to handle.

## Step 7 · Type optimisation, validation gate and DQ report

**Why this matters.** A cleaned table is only trustworthy if a **validation gate** proves it. The DQ report is your audit evidence for Model Risk Management, and the parquet file (typed, compressed) is the contract with every downstream lab.

#### ✏️ What to write in the next cell
- **Blank 13** *(inline)* — the hand-off file every later lab expects

In [ ]:
int_cols = ["age", "tenure_months", "digital_logins_30d"]
df[int_cols] = df[int_cols].round().astype("int32")
for c in ["gender", "nationality_group", "emirate", "employment_type", "employer_category"]:
    df[c] = df[c].astype("category")

def validate(d: pd.DataFrame) -> dict:
    return {
        "unique_customer_id": d.customer_id.is_unique,
        "age_in_range": d.age.between(18, 100).all(),
        "salary_positive": (d.monthly_salary_aed > 0).all(),
        "balance_non_negative": (d.avg_balance_aed >= 0).all(),
        "seven_emirates": d.emirate.nunique() == 7,
        "no_pii_columns": not {"email", "mobile"} & set(d.columns),
        "no_leakage_columns": not {"account_closed_date", "retention_offer_made"} & set(d.columns),
        "target_binary": set(d.churned_6m.unique()) <= {0, 1},
    }
checks = {k: bool(v) for k, v in validate(df).items()}; print(checks)
assert all(checks.values()), "validation gate failed"

mem = lambda d: round(d.memory_usage(deep=True).sum() / 1e6, 1)
dq_report = pd.DataFrame({
    "raw":   {"rows": len(raw), "dup_customer_ids": raw.customer_id.duplicated().sum(), "cells_missing_pct": round(raw.isna().mean().mean() * 100, 2),
              "invalid_ages": (~raw.age.between(18, 100)).sum(), "distinct_emirate_labels": raw.emirate.nunique(), "memory_mb": mem(raw)},
    "clean": {"rows": len(df), "dup_customer_ids": df.customer_id.duplicated().sum(), "cells_missing_pct": round(df.isna().mean().mean() * 100, 2),
              "invalid_ages": (~df.age.between(18, 100)).sum(), "distinct_emirate_labels": df.emirate.nunique(), "memory_mb": mem(df)},
})
dq_report.to_csv(CKPT / "dq_report.csv")
pd.Series(dropped).to_csv(CKPT / "dq_dropped_columns.csv")
df.to_parquet(____, index=False)  # ✏️ B13
dq_report

**✔ Checkpoint.** All 8 checks `True`; memory falls by ≈40–45% (category dtypes, no PII strings); `customers_clean.parquet` written (≈10 MB).

## ✅ Lab complete
You now own `customers_clean.parquet` — the first certified asset of the MIZAN platform. **Next:** Lab L1.2 builds the credit feature pipeline.